# Technical Interview: Asynchronous LLM Orchestration

### **The Scenario**
We need to process a batch of 10,000 user prompts through a third-party Generative AI API.

---

### **The Constraints**

* **Concurrency Limit:** The API has a strict rate limit. We can only have exactly 5 concurrent requests in flight at the exact same time.
* **Transient Failures:** The API is flaky and occasionally returns a `503 Service Unavailable` error. You must handle this by retrying the request using exponential backoff.
* **Fatal Failures:** The API occasionally returns a `400 Bad Request` error. This is a fatal error, meaning the prompt is malformed, and it should *not* be retried.
* **Data Persistence:** Because this is a long-running job, we cannot wait for all 10,000 tasks to finish before saving. You must write the results to a local disk (e.g., a `results.jsonl` file) the exact millisecond they complete, regardless of what order they finish in.

---

### **The Task**
Write the asynchronous Python architecture (using `asyncio`) to solve this. Use the provided `make_api_call` mock function. Focus on the structural logic, how you enforce the concurrency limit, and how you route the failure states.

In [2]:
import asyncio
import random
import json

# This is a MOCK SETUP TO SIMULATE Making an API call
# Do not modify this block.
# Use make_api_call(p) in your solution.
class APIError(Exception):
    def __init__(self, status_code):
        self.status_code = status_code

async def make_api_call(p):
    await asyncio.sleep(1) # Simulate base network latency

    roll = random.random()
    if roll < 0.1:
        raise APIError(400)  # 10% chance of a Fatal Error
    elif roll < 0.3:
        raise APIError(503)  # 20% chance of a Transient Error

    return f'Here is the answer for: {p}'

In [14]:
# Feel free to modify the function signatures as needed

async def process(p, num_try=4, multiplier=2, wait=1):
  await asyncio.sleep(wait)
  if num_try == 0:
    res = {'message': 'Max number of retries exceeded'}
    with open('result.jsonl', 'w', encoding='utf-8') as f:
      # json.dumps converts dict to a JSON string on one line
      json_record = json.dumps(res, ensure_ascii=False)
      f.write(json_record + '\n')

    #return {'message': 'Max number of retries exceeded'}
  # TODO: Implement the worker logic and retry/error handling here
  try:
    res = await make_api_call(p)
    with open('result.jsonl', 'w', encoding='utf-8') as f:
        # json.dumps converts dict to a JSON string on one line
        json_record = json.dumps(res, ensure_ascii=False)
        f.write(json_record + '\n')

  except(APIError) as e:
    if e.status_code == 400:
      res = {'message' : 'Fatal Error 400'}
      with open('result.jsonl', 'w', encoding='utf-8') as f:
        # json.dumps converts dict to a JSON string on one line
        json_record = json.dumps(res, ensure_ascii=False)
        f.write(json_record + '\n')
    else:
      await process(p, num_try=num_try-1, multiplier=multiplier, wait=wait*multiplier)

async def main():

    # Using 20 prompts for a quick test run
    prompts = [f'prompt_{i}' for i in range(20)]

    #tasks = [process(prompt) for prompt in prompts]

    # TODO: Implement concurrency control, task creation, and file streaming logic here
    semaphore = asyncio.Semaphore(5)

    async def controlled_process(p):
        async with semaphore:
            return await process(p)

    tasks = [controlled_process(p) for p in prompts]

    await asyncio.gather(*tasks)

In [15]:
await main()